# 📗 자연어 처리 — 전통 텍스트 분석

> 엔코아 AI캠퍼스 · 데이터 분석 & AI 머신러닝 캠프

지난 시간에 리뷰를 **깨끗한 단어 목록**으로 바꿨습니다. 이제 그 단어로 **분석**할 차례입니다. 무슨 말이 많이 나왔는지 세고(**빈도**), 한눈에 보이게 그리고(**워드클라우드**), 문서를 숫자 표로 바꿔(**BoW·TF-IDF**) **핵심어**를 뽑고, 긍정과 부정을 **가르는 특징어**를 찾아냅니다. 마지막으로 **함께 붙어 다니는 단어**(n-gram·동시 출현)까지 봅니다.

이 방법들은 딥러닝 이전부터 쓰였고 지금도 현장에서 **가장 먼저** 돌려 보는 도구입니다 — 빠르고, 가볍고, 결과를 사람이 바로 읽을 수 있기 때문입니다.

## ⏪ 복습 — 지난 시간: 전처리 파이프라인

지난 시간에 만든 두 함수가 이번 시간의 **입구**입니다.

- **`clean_text(text)`** — 정규표현식으로 특수문자 제거 → 반복문자 축약(`ㅋㅋㅋㅋ`→`ㅋㅋ`) → 공백 정규화.
- **`tokenize(text, stopwords)`** — 형태소 분석(`kiwi.tokenize`) → 품사 필터(`NN`·`VA`·`VV`) → 2글자 이상 → 불용어 제거.
- 불용어는 **일반**(`stopwords_ko.json` 679개) **+ 도메인**(`영화`·`평점`·`생각`·`사람`… — 빈도 상위를 눈으로 보고 직접 만들었습니다).

정제 전이라면 `최고!!!` 와 `최고` 가 다른 단어였지만, 이제는 같은 `최고` 하나로 세어집니다. **세기 시작합시다.**

**오늘의 목표**

- [ ] `Counter` 로 **단어 빈도**를 세고 긍정·부정별 상위 단어를 막대그래프로 비교한다.
- [ ] **워드클라우드**를 한글이 깨지지 않게(`font_path`) 그린다.
- [ ] **BoW(CountVectorizer)** 와 **TF-IDF(TfidfVectorizer)** 의 차이를 설명한다.
- [ ] **`min_df`·`max_df`** 로 어휘를 자동으로 거르고, `max_df` 결과를 **고빈도 불용어 후보**로 검토한다.
- [ ] 동시 출현을 **네트워크 그래프**로 그려 단어들의 관계 지도를 읽는다.
- [ ] 클래스별 평균 TF-IDF 상위어에 **공유 노이즈**가 섞이는 문제를 확인한다.
- [ ] **차이 기반 특징어**(긍정평균 − 부정평균)로 그 클래스만의 단어를 뽑는다.
- [ ] **n-gram**(bigram)과 **동시 출현**으로 함께 붙어 다니는 표현을 찾는다.
- [ ] TF-IDF 가 **무엇을 못 하는지**(같은 단어가 나와야만 비슷하다고 본다) 눈으로 확인한다.

아래 셀을 먼저 실행해 라이브러리와 한국어 형태소 분석기를 준비하세요.

In [ ]:
# [제공 코드] 이 단원에 필요한 라이브러리와 한글 폰트를 준비합니다.
import re
import json
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from kiwipiepy import Kiwi
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

sns.set_theme(style='whitegrid')
import platform

# 한글 폰트 — 실행 중인 OS 에 맞춰 자동 설정
if platform.system() == 'Windows':
    KOREAN_FONT = 'Malgun Gothic'
    FONT_PATH = 'C:/Windows/Fonts/malgun.ttf'
elif platform.system() == 'Darwin':          # macOS
    KOREAN_FONT = 'AppleGothic'
    FONT_PATH = '/System/Library/Fonts/Supplemental/AppleGothic.ttf'
else:                                        # Linux (Colab 등)
    KOREAN_FONT = 'NanumGothic'
    FONT_PATH = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'

plt.rcParams['font.family'] = KOREAN_FONT
plt.rcParams['axes.unicode_minus'] = False   # 마이너스(−) 부호 깨짐 방지

kiwi = Kiwi()   # 한국어 형태소 분석기(자바 불필요)

## 데이터 살펴보기 — 영화 리뷰 2000건

지난 시간과 같은 **영화 리뷰** 데이터입니다(`text` 본문 · `label` 1=긍정/0=부정). 새로 여는 노트북이니 데이터의 생김새를 다시 한 번 확인하고 시작합니다.

In [ ]:
reviews = pd.read_csv('data/movie_reviews.csv')

print('리뷰 크기:', reviews.shape)
print('\n[앞부분] head()')
display(reviews.head())
print('\n[감성(label) 분포]  1=긍정, 0=부정')
display(reviews['label'].value_counts().to_frame('건수'))

### 전처리 파이프라인 다시 붙이기

<img src="images/교안/전처리_파이프라인.png" width="820" style="max-width:100%"/>


지난 시간에 완성한 `clean_text` · `tokenize` 를 그대로 가져와 **리뷰 2000건을 전부 토큰으로** 바꿔 둡니다. 이 토큰 목록이 오늘 하는 모든 분석의 재료입니다.

In [ ]:
# 지난 시간에 완성한 전처리 파이프라인을 그대로 가져온다
with open('data/stopwords_ko.json', encoding='utf-8') as f:
    stopwords_general = set(json.load(f))

stopwords_domain = {'영화', '평점', '생각', '사람', '정말', '진짜', '그냥', '너무', '완전', '때문', '정도'}   # 영화 리뷰의 도메인 불용어(빈도를 보고 직접 만듬)
stopwords_all = stopwords_general | stopwords_domain


def clean_text(text):
    """원시 텍스트에서 노이즈를 걷어낸다(특수문자·반복문자·공백)."""
    text = re.sub(r'[^가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]', ' ', str(text))   # 특수문자·이모지 제거
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)              # ㅋㅋㅋㅋ→ㅋㅋ 반복 축약
    text = re.sub(r'\s+', ' ', text)                        # 공백 정규화
    return text.strip()


def tokenize(text, stopwords):
    """정제 → 형태소 → 품사 필터 → 2글자 이상 → 불용어 제거."""
    return [t.form for t in kiwi.tokenize(clean_text(text))
            if t.tag.startswith(('NN', 'VA', 'VV'))          # 명사·형용사·동사만
            and len(t.form) > 1 and t.form not in stopwords]


# 리뷰 2000건을 모두 토큰으로 바꿔 둔다(조금 걸립니다)
tokens_list = [tokenize(t, stopwords_all) for t in reviews['text']]
reviews['tokens'] = tokens_list

print('예시 — 원문 :', reviews.loc[0, 'text'])
print('예시 — 토큰 :', reviews.loc[0, 'tokens'])
print('\n토큰이 하나도 안 남은 리뷰:', sum(1 for tk in tokens_list if len(tk) == 0), '건')

---
# 1. 단어 빈도 — 무슨 말이 많이 나왔나

## 왜 필요할까요?
텍스트 분석의 **가장 첫 걸음**은 세는 것입니다. 어떤 단어가 몇 번 나왔는지만 알아도 데이터의 성격이 단번에 드러납니다. 그리고 **긍정 리뷰와 부정 리뷰를 나눠 세면**, 두 집단이 서로 다른 말을 쓴다는 게 보입니다.

### 문법 — 이미 배운 `Counter`
- **`Counter()`** 를 만들고 **`.update(단어리스트)`** 로 계속 더한다.
- **`.most_common(n)`** → `(단어, 횟수)` 짝을 빈도 순으로 n개.

긍정(`label == 1`) 리뷰의 토큰만 모아 세고, 부정(`label == 0`)도 따로 세어 **나란히 비교**합니다.

In [ ]:
# 긍정·부정 리뷰의 단어를 각각 센다
pos_counter = Counter()
neg_counter = Counter()
for tokens, label in zip(reviews['tokens'], reviews['label']):
    if label == 1:
        pos_counter.update(tokens)
    else:
        neg_counter.update(tokens)

print('긍정 리뷰의 서로 다른 단어 수:', len(pos_counter))
print('부정 리뷰의 서로 다른 단어 수:', len(neg_counter))

compare = pd.DataFrame({
    '긍정 top10': [f'{w} ({n})' for w, n in pos_counter.most_common(10)],
    '부정 top10': [f'{w} ({n})' for w, n in neg_counter.most_common(10)],
}, index=range(1, 11))
compare.index.name = '순위'
display(compare)

긍정 쪽에는 `재밌`·`최고`·`감동` 이, 부정 쪽에는 `아깝`·`쓰레기`·`재미없` 이 보입니다. 그런데 **양쪽에 똑같이 등장하는 단어**(`연기`·`나오`·`재미`)도 있습니다 — 이 점은 뒤에서 다시 다룹니다.

이제 상위 단어를 **막대그래프**로 그려 봅니다. `fig, axes = plt.subplots(...)`로 Figure와 Axes를 만든 뒤 `sns.barplot(..., ax=axes[i])`로 그립니다.

<img src="images/교안/freq_top10.png" width="820">

In [ ]:
# 긍정·부정 상위 10단어 막대그래프 (좌우로 나란히)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

pos_words = [w for w, n in pos_counter.most_common(10)]
pos_nums = [n for w, n in pos_counter.most_common(10)]
sns.barplot(x=pos_nums, y=pos_words, ax=axes[0], color='tab:blue', errorbar=None)
axes[0].set_title('긍정 리뷰 상위 10단어')
axes[0].set_xlabel('빈도')

neg_words = [w for w, n in neg_counter.most_common(10)]
neg_nums = [n for w, n in neg_counter.most_common(10)]
sns.barplot(x=neg_nums, y=neg_words, ax=axes[1], color='tab:red', errorbar=None)
axes[1].set_title('부정 리뷰 상위 10단어')
axes[1].set_xlabel('빈도')

fig.tight_layout()
plt.show()

### 따라하기 전용 데이터 — 고객센터 상담 만족도

시연은 영화 리뷰였죠. 따라하기에서는 같은 분석을 **고객센터 상담 문장 12건**에 적용합니다. 아래 제공 셀은 만족·불만 문장을 토큰화해 `support_reviews`와 `support_docs`를 준비합니다. 분석 코드는 이어지는 따라하기에서 직접 작성합니다.

In [ ]:
# [제공 코드] 따라하기 전용 고객센터 상담 데이터와 토큰을 준비합니다.
support_reviews = pd.DataFrame({
    'text': [
        '상담원이 친절하고 답변이 빨라서 만족했다',
        '배송 문의를 정확히 안내해 주어 해결됐다',
        '교환 절차를 자세히 설명해 줘서 편했다',
        '환불 처리가 빠르고 안내가 명확했다',
        '상담 연결이 빠르고 직원이 친절했다',
        '문제가 바로 해결돼서 서비스가 좋았다',
        '상담 연결이 너무 느리고 답변도 늦었다',
        '환불 안내가 불명확해서 다시 문의했다',
        '배송 지연 이유를 설명하지 않아 답답했다',
        '교환 절차가 복잡하고 처리가 늦었다',
        '직원 답변이 불친절하고 도움이 안 됐다',
        '문제가 해결되지 않아 여러 번 문의했다',
    ],
    'label': [1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
})
support_stopwords = stopwords_general | {'상담', '문의', '안내', '문제'}
support_reviews['tokens'] = [tokenize(text, support_stopwords)
                             for text in support_reviews['text']]
support_docs = [' '.join(tokens) for tokens in support_reviews['tokens']]

display(support_reviews[['text', 'label', 'tokens']].head(3))

### 🖐️ 함께 따라하기 — 상담 문장의 상위 단어 세기

영화 리뷰 시연과 다른 **고객센터 상담 문장 12건**의 단어를 세어 상위 10개를 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 빈 Counter 객체를 만들어 support_counter 변수에 담는다
# 2) support_reviews['tokens']의 각 원소(한 상담 문장의 문자열 리스트)를 돌면서
#    support_counter.update(tokens)로 단어 빈도를 누적한다
# 3) len(support_counter)로 서로 다른 단어가 몇 개인지 출력한다
# 4) support_counter.most_common(10)의 (단어, 횟수) 튜플을 출력한다

### ✅ 바로 확인 퀴즈

**1.** `Counter` 에서 빈도 상위 5개를 꺼내는 메서드는 무엇인가요?

<details><summary>정답 보기</summary>

**`.most_common(5)`** 입니다. `(단어, 횟수)` 짝의 리스트를 빈도 내림차순으로 돌려줍니다.

</details>

**2.** 긍정 top10 과 부정 top10 에 **똑같이** 들어 있는 단어(`연기` 등)는 감성을 구분하는 데 도움이 될까요?

<details><summary>정답 보기</summary>

**도움이 되지 않습니다.** 양쪽에 고르게 나오는 단어는 "이 리뷰가 긍정인지 부정인지"를 알려 주지 못합니다. 이런 단어를 걸러 내는 방법을 이번 시간 뒤쪽에서 배웁니다.

</details>

---
# 2. 워드클라우드 — 빈도를 그림 한 장으로

## 왜 필요할까요?
막대그래프는 정확하지만 10개 남짓만 보여 줍니다. **워드클라우드**는 수십·수백 단어를 한 그림에 담아 **자주 나온 말일수록 크게** 그려 줍니다. 보고서·발표에서 "이 데이터는 대충 이런 이야기"를 **한눈에** 전달할 때 강력합니다.

### 문법
- **`WordCloud(font_path=FONT_PATH, width=800, height=400, background_color='white')`** — 그림판을 만든다.
- **`.generate_from_frequencies(빈도딕셔너리)`** — `Counter` 를 그대로 넣으면 크기를 정해 그려 준다.
- 그린 결과는 `fig, ax = plt.subplots()`로 만든 뒤 **`ax.imshow(wc)`**로 띄우고 **`ax.axis('off')`**로 축을 끈다(그림이지 그래프가 아니니까).

> ### ⚠️ `font_path` 는 **필수**입니다
> 워드클라우드 라이브러리의 기본 폰트에는 **한글 글자가 없습니다**. `font_path` 를 빼면 모든 한글이 **□□□** 로 나옵니다. SETUP 셀이 **실행 중인 OS 를 감지해** 알맞은 폰트 파일 경로를 `FONT_PATH` 에 담아 두니(맥·윈도우·리눅스), 우리는 `font_path=FONT_PATH` 만 넘기면 됩니다.

긍정 리뷰와 부정 리뷰의 워드클라우드를 **나란히** 그려 비교합니다.

<img src="images/교안/wordcloud_freq.png" width="880">

In [ ]:
# 긍정·부정 워드클라우드 — font_path 를 반드시 넣는다(안 넣으면 한글이 □□□)
wc_pos = WordCloud(font_path=FONT_PATH, width=800, height=400,
                   background_color='white', colormap='Blues')
wc_pos = wc_pos.generate_from_frequencies(pos_counter)

wc_neg = WordCloud(font_path=FONT_PATH, width=800, height=400,
                   background_color='white', colormap='Reds')
wc_neg = wc_neg.generate_from_frequencies(neg_counter)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(wc_pos)
axes[0].set_title('긍정 리뷰')
axes[0].axis('off')
axes[1].imshow(wc_neg)
axes[1].set_title('부정 리뷰')
axes[1].axis('off')
fig.tight_layout()
plt.show()
print('큰 글자일수록 자주 나온 단어 — 빈도를 그대로 크기로 옮긴 그림이다')

### 🖐️ 함께 따라하기 — 상담 문장 워드클라우드

영화 리뷰 시연과 다른 **고객센터 상담 문장**의 워드클라우드를 그려 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 앞 따라하기에서 만든 support_counter Counter 변수를 사용한다
# 2) WordCloud(font_path=FONT_PATH, width=800, height=400,
#    background_color='white') 객체를 support_wc 변수에 담는다
# 3) support_wc.generate_from_frequencies(support_counter) 결과를 다시 support_wc 변수에 담는다
# 4) fig, ax = plt.subplots(figsize=(10, 5))로 Figure와 Axes를 만든다
#    ax.imshow(support_wc), ax.set_title(...), ax.axis('off')를 적용하고 plt.show()를 호출한다

### ✅ 바로 확인 퀴즈

**1.** 워드클라우드에 한글이 전부 `□□□` 로 나옵니다. 무엇을 빠뜨린 걸까요?

<details><summary>정답 보기</summary>

**`font_path`** 입니다. 기본 폰트에는 한글 글자가 없어서, 한글 폰트 파일의 경로를 `WordCloud(font_path=FONT_PATH, ...)` 로 지정해 줘야 합니다.

</details>

**2.** 워드클라우드에서 **글자 크기**는 무엇을 나타내나요?

<details><summary>정답 보기</summary>

그 단어의 **빈도(등장 횟수)** 입니다. 자주 나온 단어일수록 크게 그려집니다.

</details>

---
# 3. BoW 에서 TF-IDF 로 — 문서를 숫자 표로

## 왜 필요할까요?
지금까지는 **전체를 한 덩어리로** 세었습니다. 하지만 "**이 리뷰 한 건**의 핵심어는 무엇인가?"를 물으려면 **문서마다** 숫자를 매겨야 합니다. 그러려면 먼저 문서를 **숫자 벡터**로 바꿔야 합니다.

### ① BoW (Bag of Words) — 단어를 담은 자루
문서를 "**어떤 단어가 몇 번 나왔나**" 로만 표현합니다. 어순은 버리고 **개수만** 셉니다 — 그래서 이름이 '단어 자루'입니다.

| | 재밌 | 감동 | 쓰레기 |
|---|---|---|---|
| 리뷰 A | 2 | 1 | 0 |
| 리뷰 B | 0 | 0 | 3 |

- **`CountVectorizer(min_df=5)`** — 5개 문서 미만에 나오는 희귀 단어는 버린다(어휘 축소·잡음 제거).
- **`.fit_transform(문서리스트)`** → 어휘를 배우고 곧바로 `(문서 수, 어휘 수)` 모양의 표로 변환.

### ② 그런데 개수만 세면 생기는 문제
**흔한 단어가 표를 지배**합니다. 어느 리뷰에나 나오는 단어는 개수가 커도 그 문서의 **특징이 아닙니다.**

### ③ TF-IDF — 흔한 단어의 힘을 빼자
**TF**(Term Frequency) × **IDF**(Inverse Document Frequency)

- **TF** — 이 문서에 **자주** 나온 단어일수록 ↑
- **IDF** — **다른 문서에는 드물게** 나오는 단어일수록 ↑ (모든 문서에 다 나오면 기본 IDF가 최솟값)

> **한 줄 요약: "이 문서엔 자주 나오는데, 다른 문서엔 드문 단어"가 이 문서의 핵심어다.**

> scikit-learn 기본값은 `smooth_idf=True`라 `IDF = log((1 + 전체 문서 수) / (1 + 해당 단어가 나온 문서 수)) + 1`로 계산합니다. 모든 문서에 나온 단어도 IDF가 0이 아니라 **최솟값 1**입니다. 또한 각 문서 벡터를 기본 `norm='l2'`로 정규화하므로, 점수는 우선 한 문서 안에서의 **상대적 중요도**로 읽고 서로 다른 문서의 절대 점수를 단순 비교하지 않습니다.

- **`TfidfVectorizer(min_df=5)`** — 사용법은 CountVectorizer 와 똑같고, 값만 개수 대신 TF-IDF 점수.
- 우리는 이미 토큰화를 끝냈으므로, 토큰을 **공백으로 이어 붙인 문자열**을 넣고 `token_pattern=r'\S+'`를 지정합니다. `\S`는 공백이 아닌 문자, `+`는 1개 이상이므로 **공백으로 구분된 토큰 덩어리 하나를 한 단어로 읽으라**는 뜻입니다.

### 벡터라이저에서 꼭 구분할 세 메서드
| 메서드 | 하는 일 | 언제 쓰나 |
|---|---|---|
| `fit(docs)` | 어휘를 배우고, TF-IDF라면 IDF도 계산 | 학습 규칙만 만들 때 |
| `transform(docs)` | 이미 배운 어휘·IDF를 고정한 채 문서를 숫자로 변환 | 새 문서에 같은 기준을 적용할 때 |
| `fit_transform(docs)` | `fit` 후 같은 문서를 바로 `transform` | 처음 학습 문서 표를 만들 때 |

`get_feature_names_out()`은 행렬의 **열 순서와 정확히 같은 단어 배열**을 돌려줍니다. 따라서 `X[i, j]`는 i번 문서에서 `terms[j]` 단어의 점수입니다. 벡터라이저 결과는 0을 저장하지 않는 **희소 행렬**이라 큰 전체 행렬을 바로 밀집 배열로 바꾸지 않고, 필요한 한 행만 `X[i].toarray()[0]`으로 읽습니다.

In [ ]:
# 토큰 리스트를 공백으로 이어 붙여 '이미 전처리된 문서' 문자열로 만든다
docs = [' '.join(tokens) for tokens in reviews['tokens']]
print('문서 예시:', docs[0])

# ① BoW — 단어 개수 표
count_vec = CountVectorizer(min_df=5, token_pattern=r'\S+')
bow = count_vec.fit_transform(docs)
print('\nBoW 표 모양:', bow.shape, '  (문서 2000건 × 어휘)')

# ② TF-IDF — 같은 표지만 값이 '중요도' 점수
tfidf_vec = TfidfVectorizer(min_df=5, token_pattern=r'\S+')
X = tfidf_vec.fit_transform(docs)
terms = tfidf_vec.get_feature_names_out()
print('TF-IDF 표 모양:', X.shape, '  (문서 2000건 × 어휘 374개)')
print('어휘 수:', len(terms))

이제 **문서 한 건**의 TF-IDF 점수를 열어 봅니다. 점수가 높은 단어가 곧 **그 리뷰의 핵심어 후보**입니다.

### 점수 배열에서 상위 단어를 찾는 핵심 순서
1. `row = X[0].toarray()[0]` — 0번 문서의 점수를 1차원 배열로 꺼냅니다.
2. `np.flatnonzero(row > 0)` — 실제로 등장해 점수가 0보다 큰 열 위치만 찾습니다.
3. `np.argsort(-점수)` — 값을 돌려주는 함수가 아니라 **큰 값부터의 위치(인덱스)** 를 돌려줍니다.
4. `terms[i]` — i번째 열 위치를 실제 단어 이름으로 바꿉니다.

> 0점 위치를 먼저 제외해야 단어가 적은 문서에서 나오지 않은 단어가 상위 목록에 섞이지 않습니다.

In [ ]:
# TF-IDF 행렬
display(X.toarray())

In [ ]:
# 0번 리뷰의 TF-IDF 점수
scores = pd.Series(
    X[0].toarray()[0],
    index=terms
)

# 점수가 높은 단어 5개
top5 = scores.nlargest(5)

print('원문:', reviews.loc[0, 'text'])
print('토큰:', reviews.loc[0, 'tokens'])

print('\n[0번 리뷰의 TF-IDF 상위 5단어]')
print(top5)

## CountVectorizer vs TF-IDF — 무엇이 같고 무엇이 다른가

<img src="images/교안/TFIDF_원리.png" width="900" style="max-width:100%"/>


두 도구는 **아주 닮았습니다.** 사용법도 `fit_transform` 으로 똑같죠. 그래서 언제 무엇을 쓸지 헷갈립니다. 정리해 둡시다.

### 공통점
- 둘 다 **단어 빈도**에서 출발합니다.
- 둘 다 **문서-단어 행렬**(행=문서, 열=단어)을 만들고, 대부분의 칸이 0인 **희소** 행렬입니다.
- 어휘를 정하는 옵션(`min_df`·`ngram_range`)도 똑같이 씁니다.

### 핵심 차이
| | CountVectorizer | TfidfVectorizer |
|---|---|---|
| 칸에 들어가는 값 | 이 문서에 **몇 번** 나왔나 (TF) | TF **×** IDF |
| 흔한 단어 | 그대로 큰 값 | **IDF 가 깎아 내린다** |
| 문서마다 1번씩 나온 단어들 | 전부 **1** — 순위를 못 매김 | **드문 단어가 위로** 올라옴 |
| 기본 정규화 | 적용하지 않음 | 각 문서 벡터에 **L2 정규화** 적용 |
| 주로 쓰는 곳 | 빈도 세기·동시 출현 | **핵심어 추출**·문서 비교 |

같은 리뷰 한 건을 두 방식으로 벡터화해 **상위 단어가 어떻게 달라지는지** 직접 봅시다.

In [ ]:
# 같은 문서를 CountVectorizer 와 TfidfVectorizer 로 각각 벡터화해 상위 단어를 비교
count_terms = count_vec.get_feature_names_out()

i = 1052   # TF-IDF 차이가 잘 드러나는 리뷰
print('원문:', reviews.loc[i, 'text'])
print('토큰:', reviews.loc[i, 'tokens'])

count_row = bow[i].toarray()[0]
tfidf_row = X[i].toarray()[0]

cmp_table = pd.DataFrame({
    'Count 상위': [f'{count_terms[j]} ({int(count_row[j])})' for j in np.argsort(-count_row)[:4]],
    'TF-IDF 상위': [f'{terms[j]} ({tfidf_row[j]:.3f})' for j in np.argsort(-tfidf_row)[:4]],
}, index=range(1, 5))
cmp_table.index.name = '순위'
display(cmp_table)

print('Count 는 네 단어가 모두 1회라 순위를 못 매긴다 —')
print('TF-IDF 는 다른 리뷰엔 드문 "시리즈" 를 1위로 올려 준다. 이게 IDF 의 힘이다')

## `min_df` · `max_df` — 문서 빈도로 어휘를 자동으로 거르기

`min_df=5`가 희귀어를 거른다는 뜻을 앞에서 먼저 확인했습니다. 이제 정수·비율의 차이와 경계까지 정확히 봅시다. 두 옵션 모두 **그 단어가 등장한 문서 수(document frequency, DF)** 를 기준으로 어휘를 잘라냅니다.

<strong>기준은 "몇 번 나왔나"가 아니라 "몇 개의 문서에 나왔나"입니다.</strong> 한 리뷰에서 `영화`가 5번 나와도 문서 빈도는 1입니다.

| 설정 | 정수를 넣으면 | 0~1 소수를 넣으면 | 유지 조건 |
|---|---|---|---|
| `min_df` | 최소 문서 수 | 최소 문서 비율 | 경계와 같거나 큼 — **미만 제외** |
| `max_df` | 최대 문서 수 | 최대 문서 비율 | 경계와 같거나 작음 — **초과 제외** |

전체가 2000문서라면 다음처럼 환산됩니다.

- `min_df=5` → 5개 이상 등장한 단어 유지, 4개 이하는 제외.
- `min_df=0.01` → 1%인 20개 이상 등장한 단어 유지.
- `max_df=1000` → 1000개 이하 등장한 단어 유지.
- `max_df=0.85` → 85%인 1700개까지 유지, **1700개를 초과**하면 제외.

> <strong>자료형 주의:</strong> `1`은 정수 1문서, `1.0`은 비율 100%입니다. `max_df=1`과 `max_df=1.0`은 전혀 다른 설정입니다.

작은 코퍼스는 `min_df=2`처럼 **정수**가 해석하기 쉽고, 문서 수가 자주 달라지는 큰 코퍼스는 `min_df=0.01`처럼 **비율**이 이식하기 쉽습니다. 보통 작은 값에서 시작해 **남은 어휘 수와 제거된 단어를 직접 확인**하며 조정합니다. `max_df`를 너무 낮추면 도메인의 핵심어까지 잃을 수 있습니다.

### 경계를 미니 코퍼스로 확인하기

10문서에서 `공통`은 10개, `자주`는 8개, `가끔`은 3개, `희귀`는 1개 문서에 나오도록 만들었습니다. 정수 기준 `3~8개`와 비율 기준 `30~80%`가 같은 어휘를 남기는지 확인합니다.

> 아래의 **`binary=True`** 는 한 문서에서 같은 단어가 여러 번 나와도 값을 **1**로 만듭니다. 그래야 열의 합이 전체 등장 횟수가 아니라 **그 단어가 나온 문서 수(DF)** 가 됩니다.

In [ ]:
# 정수와 비율의 경계가 같은 결과를 만드는지 확인한다
toy_docs = [
    '공통 자주 가끔 희귀',
    '공통 자주 가끔',
    '공통 자주 가끔',
    '공통 자주',
    '공통 자주',
    '공통 자주',
    '공통 자주',
    '공통 자주',
    '공통',
    '공통',
]
toy_df_vec = CountVectorizer(binary=True, token_pattern=r'\S+')
toy_df_matrix = toy_df_vec.fit_transform(toy_docs)
toy_terms = toy_df_vec.get_feature_names_out()
toy_df = np.asarray(toy_df_matrix.sum(axis=0)).ravel()
display(pd.DataFrame({'단어': toy_terms, '문서 수': toy_df,
                      '문서 비율': toy_df / len(toy_docs)}).sort_values('문서 수', ascending=False))

int_filter = CountVectorizer(min_df=3, max_df=8, token_pattern=r'\S+')
ratio_filter = CountVectorizer(min_df=0.3, max_df=0.8, token_pattern=r'\S+')
int_filter.fit(toy_docs)
ratio_filter.fit(toy_docs)
print('정수 3~8개 유지 :', int_filter.get_feature_names_out())
print('비율 30~80% 유지:', ratio_filter.get_feature_names_out())
print('경계인 가끔(3개)·자주(8개)는 유지되고, 희귀(1개)·공통(10개)은 제외된다')

In [ ]:
# 정수 1과 비율 1.0이 얼마나 다른지 확인한다
max_one_doc = CountVectorizer(max_df=1, token_pattern=r'\S+').fit(toy_docs)
max_all_docs = CountVectorizer(max_df=1.0, token_pattern=r'\S+').fit(toy_docs)
print('max_df=1   — 1개 문서까지만 허용:', max_one_doc.get_feature_names_out())
print('max_df=1.0 — 전체 문서까지 허용  :', max_all_docs.get_feature_names_out())

### `max_df`는 자동 불용어 후보를 찾는다

지난 시간에 우리는 빈도표를 눈으로 보고 `영화`·`평점` 을 **손으로** 골라 도메인 불용어로 만들었습니다. `max_df`는 **설정한 문서 비율을 초과하는 말**을 목록 없이 걸러 냅니다. 정말 그런지, **도메인 불용어를 빼고**(일반 불용어만 적용한) 문서로 확인해 봅시다.

In [ ]:
# 도메인 불용어를 빼고(일반 불용어만) 다시 토큰화해, max_df 가 무엇을 걸러 내는지 본다
docs_general = [' '.join(tokenize(t, stopwords_general)) for t in reviews['text']]

# 각 단어가 '몇 개의 문서'에 나왔나 = 문서 빈도(document frequency)
df_vec = CountVectorizer(binary=True, min_df=5, token_pattern=r'\S+')
df_matrix = df_vec.fit_transform(docs_general)
df_terms = df_vec.get_feature_names_out()
doc_ratio = np.asarray(df_matrix.sum(axis=0)).ravel() / len(docs_general)

print('[문서 비율이 높은 단어 top5]  (전체 2000건 중 몇 %의 리뷰에 등장했나)')
for i in np.argsort(-doc_ratio)[:5]:
    print(f'  {df_terms[i]:5s} {doc_ratio[i]:.1%}')
print('\n영화 만 36% — 나머지는 6% 이하다. 확실히 혼자 튄다')

In [ ]:
# max_df 를 낮출 때 어떤 단어가 실제로 사라지는지 확인한다
base = set(df_terms)
rows = []
for max_df in [1.0, 0.3, 0.05, 0.03]:
    vec = TfidfVectorizer(min_df=5, max_df=max_df, token_pattern=r'\S+')
    vec.fit(docs_general)
    kept = set(vec.get_feature_names_out())
    removed = sorted(base - kept)
    rows.append({'max_df': max_df, '어휘 수': len(kept),
                 '제거된 단어': ', '.join(removed) if removed else '(없음)'})

display(pd.DataFrame(rows))
print('0.30에서는 영화만, 0.05에서는 영화·재밌·평점까지 제거된다')
print('0.03까지 낮추면 감동·최고 같은 핵심어도 사라진다 — 낮을수록 좋다는 뜻이 아니다')

`max_df=0.30`은 우리가 손으로 고른 `영화`를 찾아냈지만, `0.03`은 `감동`·`최고` 같은 핵심어까지 없앴습니다. 그래서 `max_df`는 **자동 불용어 확정기**가 아니라 <strong>불용어 후보를 거르는 1차 필터</strong>로 보는 편이 정확합니다.

그렇다면 도메인 불용어 목록은 필요 없을까요? **아닙니다.**

- `max_df`는 **설정한 임계값을 초과하는 말**만 잡습니다. `평점`(5.9%)·`생각`·`사람`처럼 적당히 흔하면서 의미 없는 말은 임계값에 따라 걸리지 않을 수 있습니다.
- 반대로 도메인에 따라 **핵심어인데 문서 비율이 높은** 단어도 있어, `max_df` 만 믿으면 **중요한 말을 잃을 수** 있습니다.

> **실무는 둘을 함께 씁니다 — 수동 도메인 불용어(사람의 판단) + `min_df`/`max_df`(기계의 자동 필터).**

우리 데이터는 이미 `영화` 를 **도메인 불용어로 지워 둔** 상태라, `max_df=0.85` 를 걸어도 **걸리는 단어가 없습니다** (이미 사람이 걸러 낸 뒤니까요). 확인해 봅시다.

In [ ]:
# 우리가 쓰는 문서(도메인 불용어까지 적용)에 max_df 를 걸면?
vec_both = TfidfVectorizer(min_df=5, max_df=0.85, token_pattern=r'\S+')
vec_both.fit(docs)

print('도메인 불용어만 적용 (min_df=5)          : 어휘', len(terms), '개')
print('도메인 불용어 + max_df=0.85             : 어휘', len(vec_both.get_feature_names_out()), '개')
print('\n추가로 걸러진 단어: 0개 — 사람이 이미 지웠기 때문. 두 방식은 서로를 보완한다')

### 🖐️ 함께 따라하기 — 상담 문장의 핵심어 뽑기

영화 리뷰 시연과 다른 **고객센터 상담 문장**을 TF-IDF로 벡터화하고, 5번 문장의 상위 3단어를 원문과 견주어 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) min_df=1, token_pattern=r'\S+'인 TfidfVectorizer 객체를 support_tfidf_vec 변수에 담는다
# 2) support_docs에 fit_transform을 적용한 희소 행렬을 support_X 변수에 담고,
#    어휘 배열을 support_terms 변수에 담는다
# 3) support_reviews의 5번 원문과 토큰을 출력한다
# 4) support_X[5].toarray()[0] 1차원 점수 배열을 support_row5 변수에 담는다
# 5) 0보다 큰 열 위치만 점수 내림차순으로 정렬해 최대 3개를 support_top3_idx 변수에 담는다
# 6) 각 위치 i에서 support_terms[i] 단어와 support_row5[i] 점수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** TF-IDF 점수가 높으려면 그 단어가 어떤 조건을 만족해야 하나요?

<details><summary>정답 보기</summary>

**이 문서에는 자주** 나오고(TF↑), **다른 문서에는 드물게** 나와야(IDF↑) 합니다. 모든 문서에 다 나오는 단어는 기본 IDF가 **최솟값**이 되어 상대적 가중치가 가장 작아집니다.

</details>

**2.** `min_df=5` 는 무엇을 하는 설정인가요?

<details><summary>정답 보기</summary>

**5개 문서 미만에만 나오는 희귀 단어를 어휘에서 뺍니다.** 오타나 한 번뿐인 고유명사가 표를 불필요하게 키우는 것을 막습니다.

</details>

**3.** `max_df=0.85` 는 무엇을 걸러 내나요? 그리고 그 기준은 등장 **횟수**인가요, **문서 수**인가요?

<details><summary>정답 보기</summary>

**85%를 초과하는 문서에 등장하는 단어**를 어휘에서 뺍니다 — 정확히 85%면 유지됩니다. 기준은 등장 횟수가 아니라 **그 단어가 나온 문서의 수(비율)** 입니다. 한 리뷰에 10번 나와도 문서 빈도는 1입니다.

</details>

**4.** `max_df=1`과 `max_df=1.0`은 왜 완전히 다른가요?

<details><summary>정답 보기</summary>

`1`은 정수라 **최대 1개 문서**, `1.0`은 비율이라 **최대 100%의 문서**를 뜻합니다. 문서 수 기준을 쓸지 비율 기준을 쓸지 자료형으로 구분합니다.

</details>

**5.** CountVectorizer 와 TfidfVectorizer 의 **핵심 차이**는 무엇이며, TF-IDF에 추가로 적용되는 기본 처리는 무엇인가요?

<details><summary>정답 보기</summary>

핵심 차이는 칸에 들어가는 **값**입니다. CountVectorizer 는 "이 문서에 몇 번 나왔나"(TF)를 넣고, TfidfVectorizer 는 거기에 **IDF**(다른 문서에서 얼마나 드문가)를 곱해 흔한 단어의 점수를 깎습니다. 또한 기본 설정에서는 각 문서 벡터에 **L2 정규화**도 적용합니다.

</details>

---
# 4. 클래스별 핵심어 — 그리고 그 함정

## 왜 필요할까요?
"**긍정 리뷰의 핵심어**는 무엇인가?"를 알고 싶습니다. 가장 자연스러운 방법은 **긍정 문서들의 TF-IDF 점수를 평균**내어 높은 단어를 보는 것입니다. 부정도 마찬가지로 하고요.

### 문법
- 긍정 문서만 골라 평균: **`X[y == 1].mean(axis=0)`** → 어휘마다 평균 점수 1개.
- 결과가 행렬 모양이므로 **`np.asarray(...).ravel()`** 로 1차원 배열로 편다.
- 큰 순서 상위 10개: **`np.argsort(-평균)[:10]`**

**그런데 결과를 잘 보세요.** 뭔가 이상한 점이 있습니다.

In [ ]:
# 긍정 문서들의 평균 TF-IDF, 부정 문서들의 평균 TF-IDF
y = reviews['label'].values

pos_mean = np.asarray(X[y == 1].mean(axis=0)).ravel()   # 긍정 966건의 평균
neg_mean = np.asarray(X[y == 0].mean(axis=0)).ravel()   # 부정 1034건의 평균

pos_top = [terms[i] for i in np.argsort(-pos_mean)[:10]]
neg_top = [terms[i] for i in np.argsort(-neg_mean)[:10]]

mean_table = pd.DataFrame({'긍정 평균 top10': pos_top, '부정 평균 top10': neg_top},
                          index=range(1, 11))
mean_table.index.name = '순위'
display(mean_table)

# 양쪽 top10 에 함께 들어 있는 단어는?
shared = [w for w in pos_top if w in neg_top]
print('양쪽 top10 에 다 있는 단어:', shared)

**함정이 보입니다.** `나오`·`연기`·`재미` 는 **긍정 top10 에도, 부정 top10 에도** 들어 있습니다.

- "연기가 훌륭하다" 도 `연기`, "연기가 발연기다" 도 `연기` 입니다.
- 즉 이 단어들은 **영화 리뷰라면 어느 쪽이든 자주 쓰는 말**일 뿐, **긍정만의 특징어가 아닙니다.**

TF-IDF 는 "다른 **문서**에 드문 단어"를 높게 봤을 뿐, "다른 **클래스**에 드문 단어"를 본 게 아니기 때문입니다. 지난 시간의 도메인 불용어처럼 **또 한 겹의 공유 노이즈**가 남은 셈이죠.

> **우리가 정말 원하는 건 이것입니다 — 긍정에는 많이 나오면서 부정에는 적게 나오는 단어.** 다음 절에서 그 방법을 만듭니다.

### 🖐️ 함께 따라하기 — 상담 만족·불만의 공유 노이즈 확인하기

고객센터 상담 문장의 만족·불만 평균 TF-IDF 상위어를 구해, **양쪽에 모두 들어 있는 단어**를 뽑아 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) support_reviews['label']의 정수 배열을 support_y 변수에 담는다
# 2) support_X에서 label 1과 0의 열별 평균을 각각 support_pos_mean, support_neg_mean 배열에 담는다
# 3) 두 평균에서 상위 10개 단어를 support_pos10, support_neg10 리스트 변수에 담는다
# 4) 두 리스트에 모두 있는 단어를 support_both, 만족 목록에만 있는 단어를 support_only_pos에 담는다
# 5) support_both와 support_only_pos 리스트를 각각 출력한다

### ✅ 바로 확인 퀴즈

**1.** `연기` 가 긍정 평균 top10 과 부정 평균 top10 에 **둘 다** 들어 있습니다. 이 단어로 리뷰의 감성을 맞힐 수 있을까요?

<details><summary>정답 보기</summary>

**없습니다.** 양쪽에 고르게 나오는 단어라 "`연기` 가 있으니 긍정" 같은 판단이 불가능합니다. 감성을 가르는 힘이 없는 **공유 노이즈**입니다.

</details>

**2.** TF-IDF 는 "흔한 단어의 힘을 뺀다"고 했는데, 왜 `연기` 같은 공유 노이즈가 남았을까요?

<details><summary>정답 보기</summary>

TF-IDF 의 IDF 는 **문서 전체**를 기준으로 흔한 정도를 잽니다. `연기` 는 전체 2000건 중 일부에만 나오므로 IDF 가 그렇게 낮지 않습니다. 하지만 **긍정과 부정 양쪽에 고르게** 나오죠 — TF-IDF 는 **클래스**를 전혀 보지 않기 때문에 이 점을 잡아내지 못합니다.

</details>

---
# 5. 차이 기반 특징어 추출 — 그 클래스만의 단어

## 왜 필요할까요?
앞 절에서 확인했습니다 — **평균이 높은 단어 ≠ 그 클래스의 특징어**. 양쪽에서 다 높은 단어는 특징이 될 수 없습니다. 그렇다면 **양쪽의 차이**를 보면 됩니다.

## 비유
쌍둥이의 사진을 구별할 때, 둘 다 가진 특징(검은 머리·같은 교복)은 도움이 안 됩니다. **한쪽에만 있는 것**(점·안경)을 찾아야 하죠. 단어도 똑같습니다.

### 방법 — 빼기 한 번이면 끝
```
diff = pos_mean - neg_mean
```
- **`diff` 가 큰(양수) 단어** → 긍정에는 많은데 부정에는 적다 = **긍정만의 특징어**
- **`diff` 가 작은(음수) 단어** → 부정에는 많은데 긍정에는 적다 = **부정만의 특징어**
- **양쪽에 고르게 나오는 공유 노이즈는 빼면 서로 상쇄되어 0 근처로** 내려앉습니다 — **저절로 걸러집니다.**

정렬 방향만 바꾸면 두 목록이 한 번에 나옵니다 — `np.argsort(-diff)`(내림차순 = 긍정), `np.argsort(diff)`(오름차순 = 부정).

In [ ]:
# 차이 기반 특징어 — 긍정 평균에서 부정 평균을 뺀다
diff = pos_mean - neg_mean

feat_pos = [terms[i] for i in np.argsort(diff, descending=True)[:10]]   # 내림차순 = 긍정만의 특징어
feat_neg = [terms[i] for i in np.argsort(diff, descending=False)[:10]]    # 오름차순 = 부정만의 특징어

feat_table = pd.DataFrame({'긍정 특징어 top10': feat_pos, '부정 특징어 top10': feat_neg},
                          index=range(1, 11))
feat_table.index.name = '순위'
display(feat_table)

**공유 노이즈가 줄었습니다.** 앞 절에서 양쪽에 다 있던 `나오`·`연기`·`재미` 가 두 목록 어디에도 없습니다 — 빼기 과정에서 서로 상쇄됐기 때문입니다. 남은 단어는 한 클래스와 **더 강하게 연관된 특징어 후보**입니다.

- **긍정 특징어**: `재밌`·`최고`·`감동`·`재미있`·`사랑`·`마음`·`아름답`·`명작`·`슬프`·`여운`
- **부정 특징어**: `아깝`·`쓰레기`·`재미없`·`만들`·`최악`·`감독`·`짜증`·`실망`·`스토리`·`여자`

`감독`·`스토리`·`만들`이 부정 쪽 후보로 나온 것은 흥미롭지만, 단어 하나만으로 이유나 문맥을 확정할 수는 없습니다. 예를 들어 `연기`는 "훌륭하다"와 "별로다" 양쪽 문맥에 쓰입니다. **후보 단어가 들어간 원문을 다시 확인한 뒤** 해석해야 하며, 차이 점수는 인과나 분류 성능을 보장하지 않습니다.

아래는 **평균 방식 vs 차이 방식**을 나란히 놓은 비교표입니다.

In [ ]:
# 평균 방식과 차이 방식을 나란히 비교 — 공유 노이즈가 어디로 갔나
compare_method = pd.DataFrame({
    '긍정: 평균 방식': pos_top,
    '긍정: 차이 방식': feat_pos,
    '부정: 평균 방식': neg_top,
    '부정: 차이 방식': feat_neg,
}, index=range(1, 11))
compare_method.index.name = '순위'
display(compare_method)

print('평균 방식에 남아 있던 공유 노이즈:', shared)
print('그 단어들이 차이 방식 목록에 남아 있나?',
      [w for w in shared if w in feat_pos or w in feat_neg])

이 특징어로 **워드클라우드를 다시 그리면** 그림이 훨씬 선명해집니다. 빈도 워드클라우드에는 양쪽에 다 나오던 말이 크게 박혀 있었지만, 특징어 워드클라우드에는 **감성을 가르는 말만** 남습니다. 크기는 `diff` 의 크기(절댓값)로 정합니다.

<img src="images/교안/wordcloud_diff.png" width="880">

In [ ]:
# 특징어로 다시 그린 워드클라우드 — 크기는 차이(diff)의 크기로
pos_weights = {terms[i]: float(diff[i]) for i in np.argsort(-diff)[:60]}
neg_weights = {terms[i]: float(-diff[i]) for i in np.argsort(diff)[:60]}

wc_fp = WordCloud(font_path=FONT_PATH, width=800, height=400,
                  background_color='white', colormap='Blues')
wc_fp = wc_fp.generate_from_frequencies(pos_weights)

wc_fn = WordCloud(font_path=FONT_PATH, width=800, height=400,
                  background_color='white', colormap='Reds')
wc_fn = wc_fn.generate_from_frequencies(neg_weights)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(wc_fp)
axes[0].set_title('긍정 특징어 (차이 기반)')
axes[0].axis('off')
axes[1].imshow(wc_fn)
axes[1].set_title('부정 특징어 (차이 기반)')
axes[1].axis('off')
fig.tight_layout()
plt.show()
print('빈도 워드클라우드보다 훨씬 또렷하다 — 양쪽에 다 나오던 말이 사라졌기 때문')

### 한 장으로 합치기 — 색으로 긍정·부정을 구분한 워드클라우드

그림 두 장을 나란히 놓는 대신, **한 장에 모두 담고 색으로 편을 가를** 수도 있습니다. 보고서에 넣기 좋은 형태죠.

- 긍정 특징어(`diff` **양수**)와 부정 특징어(`diff` **음수** → `abs` 로 크기만 취함)를 **하나의 딕셔너리로 합쳐** 그린다.
- 그린 뒤 **`wc.recolor(color_func=...)`** 로 단어마다 색을 다시 칠한다 — 긍정이면 **초록**, 부정이면 **빨강**.
- 어느 편인지 기억해 둬야 하므로, 긍정 단어 집합을 미리 만들어 두고 색 함수가 그것을 참조하게 합니다.
- 범례는 **`matplotlib.patches.Patch`** 로 직접 만들어 붙입니다.

<img src="images/교안/wordcloud_signed.png" width="700">

In [ ]:
# 긍정·부정 특징어를 한 장에 — 크기는 diff 의 크기, 색은 편(긍정 초록 / 부정 빨강)
from matplotlib.patches import Patch

signed_weights = {}
for i in np.argsort(diff, descending=True)[:40]:      # 긍정 특징어 40개 (diff 양수)
    signed_weights[terms[i]] = float(abs(diff[i]))
for i in np.argsort(diff, descending=False)[:40]:       # 부정 특징어 40개 (diff 음수 → 크기만)
    signed_weights[terms[i]] = float(abs(diff[i]))

positive_words = {terms[i] for i in np.argsort(-diff)[:40]}   # 어느 편인지 기억


def sentiment_color(word, **kwargs):
    """긍정 특징어면 초록, 부정 특징어면 빨강으로 칠한다."""
    return '#2E8B57' if word in positive_words else '#DC143C'


wc_signed = WordCloud(font_path=FONT_PATH, width=900, height=450,
                      background_color='white')
wc_signed = wc_signed.generate_from_frequencies(signed_weights)
wc_signed = wc_signed.recolor(color_func=sentiment_color)   # 단어마다 색을 다시 칠한다

fig, ax = plt.subplots(figsize=(11, 6))
ax.imshow(wc_signed)
ax.set_title('영화 리뷰 특징어 — 긍정(초록) vs 부정(빨강)')
ax.axis('off')
ax.legend(handles=[Patch(color='#2E8B57', label='긍정 특징어'),
                    Patch(color='#DC143C', label='부정 특징어')],
           loc='lower right')
plt.show()
print('한 장으로 두 편을 함께 보여 준다 — 크기는 치우친 정도, 색은 어느 편인지')

### 🖐️ 함께 따라하기 — 상담 만족·불만 특징어를 점수와 함께 보기

고객센터 상담 문장의 `support_diff` 값을 **점수까지** 함께 출력해, 어떤 단어가 만족·불만 쪽으로 얼마나 치우쳐 있는지 확인해 봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) support_pos_mean - support_neg_mean 결과를 support_diff 1차원 배열에 담는다
# 2) support_diff 내림차순 상위 5개 위치를 support_pos5_idx 정수 인덱스 배열에 담는다
#    각 위치의 support_terms 단어와 support_diff 점수를 소수 4자리로 출력한다
# 3) support_diff 오름차순 상위 5개 위치를 support_neg5_idx 배열에 담고 같은 형식으로 출력한다
# 4) support_neg5_idx가 가리키는 점수의 부호를 확인한다

### ✅ 바로 확인 퀴즈

**1.** `diff = pos_mean - neg_mean` 에서 **값이 큰 음수**인 단어는 어떤 단어인가요?

<details><summary>정답 보기</summary>

**부정 리뷰의 특징어**입니다. 부정 평균이 긍정 평균보다 훨씬 크다는 뜻이니까요. 그래서 부정 특징어는 `diff` 를 **오름차순**으로 정렬해 뽑습니다.

</details>

**2.** 양쪽에 고르게 나오는 `연기` 같은 단어의 `diff` 값은 어떻게 될까요?

<details><summary>정답 보기</summary>

긍정 평균과 부정 평균이 비슷하므로 빼면 **0 근처**가 됩니다. 그래서 양쪽 상위 목록 어디에도 오르지 못하고 **저절로 걸러집니다** — 이것이 차이 기반 방식의 핵심입니다.

</details>

---
# 6. n-gram — 단어 하나로는 못 잡는 표현

## 왜 필요할까요?
BoW 는 어순을 버립니다. 그래서 `배우` 와 `연기` 가 **바로 붙어 나왔다**는 사실을 잃어버립니다. `킬링 타임` 처럼 **두 단어가 붙어야 뜻이 생기는 표현**도 못 잡죠. 그래서 **연속된 n개의 단어**를 하나의 항목으로 세는 것이 **n-gram** 입니다.

| 이름 | 세는 단위 | 예 ("배우 연기 최고") |
|---|---|---|
| unigram (1-gram) | 단어 1개 | `배우`, `연기`, `최고` |
| **bigram (2-gram)** | 연속된 단어 2개 | `배우 연기`, `연기 최고` |
| trigram (3-gram) | 연속된 단어 3개 | `배우 연기 최고` |

### 문법
- **`CountVectorizer(ngram_range=(2, 2), min_df=5, token_pattern=r'\S+')`** — `ngram_range=(2, 2)` 는 "**2개짜리만**" 세라는 뜻입니다(`(1, 2)` 면 1개와 2개를 함께).
- 전체 등장 횟수는 표를 **열 방향으로 합**하면 됩니다: `bigram_matrix.sum(axis=0)`

In [ ]:
# bigram — 연속된 두 단어를 하나로 세기
bigram_vec = CountVectorizer(ngram_range=(2, 2), min_df=5, token_pattern=r'\S+')
bigram_matrix = bigram_vec.fit_transform(docs)
bigram_terms = bigram_vec.get_feature_names_out()

bigram_counts = np.asarray(bigram_matrix.sum(axis=0)).ravel()   # 열별 합 = 전체 등장 횟수
print('bigram 개수:', len(bigram_terms))

print('\n[가장 많이 나온 bigram top10]')
for i in np.argsort(-bigram_counts)[:10]:
    print(f'  {bigram_terms[i]:12s} {bigram_counts[i]}회')

`배우 연기`(15회)·`킬링 타임`(9회)·`인생 최고`(6회)·`남자 주인공`(5회) — **단어 하나씩 볼 때는 안 보이던 표현**들입니다. 특히 `킬링 타임` 은 두 단어를 따로 보면 뜻이 완전히 달라지죠. 리뷰에서 반복되는 **관용 표현**을 찾을 때 n-gram 이 가장 빠른 길입니다.

## 실무에선 어떤 범위를 쓸까 — n-gram 선택 가이드

<img src="images/교안/ngram_동시출현.png" width="900" style="max-width:100%"/>

> 오른쪽 절반(동시 출현)은 바로 다음 절에서 다룹니다 — n-gram 은 **이웃한** 낱말을, 동시 출현은 **같은 문서 안** 낱말을 본다는 점만 먼저 구분해 두세요.


| 설정 | 이름 | 추출 단위 | 주요 활용 |
|---|---|---|---|
| `(1, 1)` | Unigram | 단일 단어 | 워드클라우드·기본 빈도 분석(탐색) |
| `(2, 2)` | Bigram | 2단어 조합 | 문맥·의미 있는 표현 포착(주제 분석) |
| **`(1, 2)`** | **Unigram + Bigram** | **단일 + 조합** | 단일어와 조합을 함께 볼 때 쓰는 **출발점 후보** |

- **탐색**할 땐 `(1, 1)`, **문맥**을 잡을 땐 `(2, 2)`, `(1, 2)`는 단어 하나의 정보(`재밌`)와 두 단어 표현(`킬링 타임`)을 **한 번에** 가져가는 흔한 출발점입니다. 다만 최종 범위는 어휘 수·결과 품질·평가 지표를 비교해 데이터에 맞게 정하세요.
- `(1, 3)` 이상으로 키우면 항목 수가 **폭발**합니다(3단어 조합은 대부분 딱 한 번만 나옵니다). 표만 커지고 얻는 건 적어 **효율이 떨어집니다.**

세 설정으로 각각 벡터화해 **어휘 수가 어떻게 달라지는지** 직접 확인해 봅시다.

In [ ]:
# 같은 문서를 세 가지 ngram_range 로 벡터화해 어휘 수를 비교
rows = []
for ngram_range in [(1, 1), (2, 2), (1, 2)]:
    vec = CountVectorizer(ngram_range=ngram_range, min_df=5, token_pattern=r'\S+')
    vec.fit(docs)
    feature_names = vec.get_feature_names_out()
    rows.append({'ngram_range': str(ngram_range), '어휘 수': len(feature_names),
                 '예시': ', '.join(feature_names[:3])})

display(pd.DataFrame(rows))
print('(1,1) 374개 + (2,2) 9개 = (1,2) 383개 — (1,2)는 둘을 함께 보는 출발점 후보다')

### 🖐️ 함께 따라하기 — 불만 상담 문장의 bigram 보기

영화 리뷰 시연과 다른 **불만 상담 문장(label=0)** 만 골라 bigram 상위 5개를 뽑아 봅니다. 문장이 6건뿐이므로 `min_df=1`로 설정합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) support_docs와 support_reviews['label']을 zip으로 묶고 label이 0인 문자열만
#    support_neg_docs 리스트 변수에 담는다
# 2) ngram_range=(2, 2), min_df=1, token_pattern=r'\S+'인 CountVectorizer 객체를
#    support_bigram_vec 변수에 담는다
# 3) fit_transform 결과를 support_bigram 희소 행렬에, 어휘 배열을 support_bigram_terms에 담는다
# 4) 열별 합계를 support_bigram_counts 1차원 배열에 담는다
# 5) 횟수가 큰 상위 5개 위치에서 bigram과 횟수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `ngram_range=(2, 2)` 와 `ngram_range=(1, 2)` 는 어떻게 다른가요?

<details><summary>정답 보기</summary>

`(2, 2)` 는 **두 단어짜리만** 셉니다. `(1, 2)` 는 **한 단어짜리와 두 단어짜리를 모두** 셉니다.

</details>

**2.** `킬링` 과 `타임` 을 따로 세면 왜 부족한가요?

<details><summary>정답 보기</summary>

`킬링 타임` 은 **두 단어가 붙어야 "시간 때우기용 영화"라는 뜻**이 생기는 표현이기 때문입니다. 따로 세면 각각의 뜻만 남고 원래 표현의 의미가 사라집니다.

</details>

---
# 7. 동시 출현 — 함께 등장하는 단어 찾기 (연관 분석)

## 왜 필요할까요?
n-gram 은 **바로 옆에 붙어야** 잡힙니다. 하지만 "배우들의 연기가 좋았고 스토리도 탄탄했다" 처럼 **한 리뷰 안 어딘가에 함께 나오는** 단어 쌍도 중요한 정보입니다. 어떤 단어들이 **같은 문서에 자주 같이 나오는지** 세는 것이 **동시 출현(co-occurrence) 분석**입니다.

## 방법 — 있으면 1, 없으면 0
1. **`CountVectorizer(binary=True, min_df=10)`** — 몇 번 나왔든 상관없이 **문서에 등장했나(1) 안 했나(0)** 만 표시.
2. 이 이진 표를 `B`라고 하면 **자기 자신과 곱합니다**: `M = (B.T @ B).toarray()`
   - `B.T @ B` 의 (i, j) 칸 = **단어 i 와 단어 j 가 함께 등장한 문서 수**. (0/1 표라서 곱하면 곧 개수가 됩니다.)
3. **`np.fill_diagonal(M, 0)`** — 대각선은 "자기 자신과 함께 나온 횟수"라 의미가 없으니 0으로.
4. 표는 **대칭**(i↔j 가 같은 값)이므로 `np.triu_indices_from(M, k=1)`로 위쪽 삼각형만 봅니다. `k=1`은 대각선을 제외한다는 뜻입니다.

> `min_df=10` — 10개 문서 미만에 나오는 단어는 뺍니다. 어휘가 작아야 단어×단어 표가 다룰 만한 크기가 됩니다.

In [ ]:
# 동시 출현 행렬 만들기
binary_vec = CountVectorizer(binary=True, min_df=10, max_df=0.85, token_pattern=r'\S+')   # 문서에 등장했나(0/1)
binary_matrix = binary_vec.fit_transform(docs)
co_terms = binary_vec.get_feature_names_out()
print('동시 출현에 쓸 어휘 수:', len(co_terms))

co_matrix = (binary_matrix.T @ binary_matrix).toarray()   # 단어×단어 동시출현 행렬
np.fill_diagonal(co_matrix, 0)                            # 자기 자신은 제외
print('행렬 모양:', co_matrix.shape)

# 위쪽 삼각형에서 상위 단어쌍 뽑기 (표가 대칭이라 아래쪽은 같은 쌍의 중복)
rows, cols = np.triu_indices_from(co_matrix, k=1)
values = co_matrix[rows, cols]

print('\n[함께 등장한 단어쌍 top10]')
for k in np.argsort(-values)[:10]:
    print(f'  ({co_terms[rows[k]]}, {co_terms[cols[k]]}) : {values[k]}건의 리뷰에 함께 등장')

`(배우, 연기)`가 21건으로 많고 `(스토리, 연기)`·`(감동, 재미)`가 뒤를 잇습니다. 이는 두 단어가 같은 리뷰에 자주 등장했다는 뜻이지, 같은 뜻이거나 인과관계가 있다는 뜻은 아닙니다. 원래 자주 나오는 단어가 유리하고 `min_df`·상위 쌍 개수에 따라 결과가 달라질 수 있으므로 원문과 함께 해석합니다.

상위 단어들만 골라 **히트맵**으로 그리면 관계가 한눈에 들어옵니다.

<img src="images/교안/cooccurrence_heatmap.png" width="640">

In [ ]:
# 동시 출현이 많은 상위 12개 단어만 골라 히트맵으로
top_words = np.argsort(-co_matrix.sum(axis=1))[:12]        # 다른 단어와 많이 어울린 단어들
sub_matrix = co_matrix[np.ix_(top_words, top_words)]
labels = [co_terms[i] for i in top_words]

fig, ax = plt.subplots(figsize=(8, 6.5))
sns.heatmap(sub_matrix, xticklabels=labels, yticklabels=labels,
            annot=True, fmt='d', cmap='YlOrRd', cbar_kws={'label': '함께 등장한 리뷰 수'}, ax=ax)
ax.set_title('단어 동시 출현 히트맵 (상위 12단어)')
fig.tight_layout()
plt.show()
print('색이 진할수록 두 단어가 같은 리뷰에 자주 함께 나온다')

## 네트워크로 보기 — 단어들의 관계 지도

히트맵은 **모든 쌍**을 빠짐없이 보여 주지만, 표가 커지면 **관계의 모양**이 눈에 안 들어옵니다. **네트워크 그래프**는 단어를 **점(노드)** 으로, 함께 나온 관계를 **선(엣지)** 으로 그려서 "누가 중심인가"·"어떤 무리가 있나"를 한눈에 보여 줍니다.

### 그림 읽는 법
| 요소 | 뜻 |
|---|---|
| **노드(점) 크기** | **연결 강도** — 그 단어가 다른 단어들과 함께 나온 횟수의 합. 클수록 대화의 **중심** |
| **엣지(선) 두께** | 두 단어가 **함께 나온 리뷰 수**. 두꺼울수록 **끈끈한 짝** |
| 노드 사이 거리 | 연결을 고려한 자동 배치 결과. 가까운 경향은 있지만 강도의 정확한 척도는 아님 |

그리는 코드는 아래에 **`# [제공 코드]`** 로 드립니다 — **내용은 이해하지 않아도 됩니다. 호출만** 하세요. 우리가 할 일은 **`(단어1, 단어2, 횟수)` 목록을 만들어 넘기는 것**뿐입니다.

In [ ]:
# [제공 코드] 동시 출현 네트워크를 그려 주는 함수입니다 — 내용은 이해하지 않아도 됩니다. 호출만 하세요.
import networkx as nx

def draw_cooccurrence_network(pairs, top_n=25, title='동시 출현 네트워크',
                              node_color='#6aa9e9', save_path=None):
    """pairs = [(단어1, 단어2, 동시출현횟수), ...] 를 네트워크로 그립니다."""
    G = nx.Graph()
    for w1, w2, cnt in pairs[:top_n]:
        G.add_edge(w1, w2, weight=int(cnt))
    strength = {n: sum(d['weight'] for _, _, d in G.edges(n, data=True)) for n in G.nodes()}
    lo, hi = min(strength.values()), max(strength.values())
    rng = (hi - lo) or 1
    sizes = [300 + (strength[n] - lo) / rng * 2200 for n in G.nodes()]
    mx = max(d['weight'] for _, _, d in G.edges(data=True))
    widths = [0.5 + G[u][v]['weight'] / mx * 4 for u, v in G.edges()]
    pos = nx.spring_layout(G, k=0.7, seed=42)
    fig, ax = plt.subplots(figsize=(11, 8))
    nx.draw_networkx_edges(G, pos, width=widths, alpha=0.35, edge_color='#999999', ax=ax)
    nx.draw_networkx_nodes(G, pos, node_size=sizes, node_color=node_color, alpha=0.9,
                           edgecolors='white', linewidths=1.5, ax=ax)
    nx.draw_networkx_labels(G, pos, font_family=KOREAN_FONT, font_size=11, ax=ax)
    ax.set_title(title, fontsize=14)
    ax.axis('off')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=110, bbox_inches='tight')
    plt.show()
    return G

먼저 함수에 넘길 **단어쌍 목록**을 만듭니다. 앞에서 상위 쌍을 뽑을 때 쓴 **상삼각 순회**를 그대로 재사용하되, 이번엔 출력 대신 **`(단어1, 단어2, 횟수)` 튜플의 리스트**로 모읍니다.

<img src="images/교안/cooccurrence_network.png" width="720">

In [ ]:
# 상삼각을 순회하며 (단어1, 단어2, 횟수) 목록을 만든다 — 횟수 내림차순
pairs = []
for k in np.argsort(-values):
    if values[k] == 0:
        break
    pairs.append((co_terms[rows[k]], co_terms[cols[k]], values[k]))

print('만들어진 단어쌍:', len(pairs), '개')
print('상위 5개:', pairs[:5])

# 제공 함수에 넘기기만 하면 된다
graph = draw_cooccurrence_network(pairs, top_n=25, title='영화 리뷰 — 단어 동시 출현 네트워크 (상위 25쌍)')
print('노드(단어)', graph.number_of_nodes(), '개 / 엣지(관계)', graph.number_of_edges(), '개')
print('배우-연기 를 중심으로 스토리·연출·감독이 이어진 덩어리가 보인다')

### 긍정 리뷰와 부정 리뷰의 네트워크는 다를까

**같은 방법을 그룹별로** 돌려 보면 됩니다. 동시 출현 행렬은 `등장표.T @ 등장표` 였으니, **그 그룹의 문서 행만 골라** 같은 곱셈을 하면 그 그룹의 동시 출현이 나옵니다.

```
group_dtm = binary_matrix[y == 1]          # 긍정 문서 행만 고른다
group_matrix = (group_dtm.T @ group_dtm)   # 그 그룹 안에서의 동시 출현
```

어휘(`co_terms`)는 전체 기준 그대로라서 **두 네트워크를 같은 잣대로 비교**할 수 있습니다.

<img src="images/교안/cooccurrence_network_pos.png" width="700">

<img src="images/교안/cooccurrence_network_neg.png" width="700">

In [ ]:
# 그룹(긍정/부정)별 동시 출현 네트워크 — 같은 어휘, 다른 문서
def build_pairs(group_mask):
    """그 그룹의 문서만 골라 동시 출현 단어쌍 목록을 만든다."""
    group_dtm = binary_matrix[group_mask]
    group_matrix = (group_dtm.T @ group_dtm).toarray()
    np.fill_diagonal(group_matrix, 0)
    group_values = group_matrix[rows, cols]        # 앞에서 만든 상삼각 위치를 재사용
    result = []
    for k in np.argsort(-group_values):
        if group_values[k] == 0:
            break
        result.append((co_terms[rows[k]], co_terms[cols[k]], group_values[k]))
    return result


pos_pairs = build_pairs(y == 1)
neg_pairs = build_pairs(y == 0)
print('긍정 상위 5쌍:', pos_pairs[:5])
print('부정 상위 5쌍:', neg_pairs[:5])

draw_cooccurrence_network(pos_pairs, top_n=20, title='긍정 리뷰의 동시 출현 네트워크',
                          node_color='#7ec98f')
draw_cooccurrence_network(neg_pairs, top_n=20, title='부정 리뷰의 동시 출현 네트워크',
                          node_color='#e98a8a')

**두 네트워크가 다른 이야기를 합니다.**

- **긍정**: `(드라마, 최고)`·`(작품, 최고)`·`(인생, 최고)` — **`최고` 가 여러 단어와 이어진 허브**입니다. 좋았던 관객은 "인생 최고의 작품"처럼 **찬사를 여기저기 붙입니다.**
- **부정**: `(배우, 연기)`(13)·`(스토리, 연기)` — **`연기` 가 허브**입니다. 실망한 관객은 **연기와 스토리를 콕 집어 지적**합니다.

전체 네트워크만 봤다면 `(배우, 연기)` 가 압도적이라 "관객은 연기 이야기를 많이 한다"에서 끝났을 겁니다. **그룹을 나눠 보니 그 이야기가 주로 부정 리뷰에서 나온다**는 것까지 알게 됐습니다.

### 🖐️ 함께 따라하기 — 상담 문장의 동시 출현 네트워크

영화 리뷰 시연과 다른 **고객센터 상담 문장**을 0/1 문서-단어 행렬로 바꾸고, 상위 단어쌍으로 네트워크를 그립니다. 제공 함수 `draw_cooccurrence_network`는 그대로 사용합니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) binary=True, min_df=1, token_pattern=r'\S+'인 CountVectorizer로 support_docs를 변환한다
#    행렬은 support_binary, 어휘 배열은 support_co_terms 변수에 담는다
# 2) support_binary의 전치행렬과 원래 행렬을 곱해 단어 동시 출현 횟수 행렬을 만든다
#    자기 자신과의 동시 출현은 0으로 바꾼다
# 3) 중복되지 않는 단어쌍을 뽑아 횟수 내림차순으로 정렬하고 support_pairs 리스트에 담는다
# 4) support_pairs 상위 15쌍을 draw_cooccurrence_network에 넘겨 support_graph 변수에 담는다
# 5) support_graph의 노드 수와 엣지 수를 출력한다

### 🖐️ 함께 따라하기 — `답변`과 가장 자주 함께 나온 단어

고객센터 상담 문장에서 `답변`과 **같은 문장에 가장 자주 함께 나온** 단어 5개를 찾아봅니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) list(support_co_terms).index('답변')으로 정수 열 위치를 support_word_idx 변수에 담는다
# 2) support_co_matrix의 해당 행을 support_word_row 1차원 배열에 담는다
# 3) 값이 큰 상위 5개 위치를 support_related_idx 정수 인덱스 배열에 담는다
# 4) 각 위치에서 support_co_terms 단어와 동시 출현 횟수를 출력한다

### ✅ 바로 확인 퀴즈

**1.** `CountVectorizer(binary=True)` 로 만든 표의 값은 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

그 단어가 그 문서에 **등장했으면 1, 안 했으면 0** 입니다. 몇 번 나왔는지는 세지 않습니다. "함께 나온 **문서 수**"를 세려면 횟수가 아니라 등장 여부만 필요하기 때문입니다.

</details>

**2.** 동시 출현 행렬에서 `np.fill_diagonal(M, 0)` 을 왜 할까요?

<details><summary>정답 보기</summary>

대각선 칸은 "단어가 **자기 자신과** 함께 나온 횟수"(= 그 단어가 나온 문서 수)라 **의미가 없고 값도 가장 커서**, 그냥 두면 상위 쌍을 뽑을 때 전부 자기 자신 쌍이 나옵니다. 그래서 0으로 지웁니다.

</details>

**3.** 네트워크 그래프에서 **노드가 크다**는 것과 **엣지가 두껍다**는 것은 각각 무엇을 뜻하나요?

<details><summary>정답 보기</summary>

**노드 크기 = 연결 강도** — 그 단어가 다른 단어들과 함께 나온 횟수의 합입니다. 크면 그 단어가 대화의 **중심**입니다. **엣지 두께 = 두 단어가 함께 나온 리뷰 수** — 두꺼울수록 **끈끈한 짝**입니다.

</details>

### ✅ 바로 확인 퀴즈

**1.** TF-IDF 벡터가 "희소하다"는 말은 무슨 뜻인가요?

<details><summary>정답 보기</summary>

벡터가 **길지만 대부분의 칸이 0** 이라는 뜻입니다. 어휘가 374개라 벡터도 374칸인데, 문서 하나에 들어 있는 단어는 몇 개뿐이라 나머지는 전부 0입니다(실제로 0의 비율이 99% 이상).

</details>

**2.** "긴장감이 넘친다" 와 "시간 가는 줄 모른다" 는 사람에겐 비슷한 말입니다. 오늘 배운 방식은 왜 이 둘을 비슷하다고 보지 못할까요?

<details><summary>정답 보기</summary>

**같은 단어가 하나도 겹치지 않기 때문**입니다. 오늘 배운 방식은 단어의 **글자 모양**만 세고 **뜻**은 모릅니다. 겹치는 단어가 없으면 두 문서는 완전히 남남으로 취급됩니다.

</details>

---
## 🚀 응용 클론코딩 — 고객센터 상담 핵심어 리포트

오늘 배운 것을 하나로 묶어 **리포트 한 장**을 만듭니다. 위에서 이미 구해 둔 값들을 모아 **보고서 형태로 출력**하면 됩니다.

| 항목 | 내용 |
|---|---|
| 데이터 요약 | 상담 문장 수, 만족·불만 건수, 어휘 수 |
| 만족 특징어 top5 | 차이 기반(`support_diff` 내림차순) |
| 불만 특징어 top5 | 차이 기반(`support_diff` 오름차순) |
| 대표 표현 top3 | bigram 상위 |
| 함께 나오는 단어쌍 top3 | 동시 출현 상위 |

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 위 따라하기에서 만든 support_reviews, support_y, support_terms, support_diff,
# support_bigram_terms, support_bigram_counts, support_pairs 변수를 그대로 사용한다
# 1) 리포트 제목 줄을 출력한다
# 2) 전체·만족·불만 문장 수와 TF-IDF 어휘 수를 출력한다
# 3) support_diff를 정렬한 위치를 support_terms에 매핑해 만족·불만 특징어 top5를 출력한다
# 4) support_bigram_counts 상위 3개 위치의 bigram과 횟수를 출력한다
# 5) support_pairs 리스트의 앞 3개 단어쌍과 동시 출현 문장 수를 출력한다

---
## 이번 강의 정리

| 분석 | 무엇을 알려 주나 | 도구 |
|---|---|---|
| 단어 빈도 | 무슨 말이 많이 나왔나 | `Counter.most_common` |
| 워드클라우드 | 빈도를 그림 한 장으로 | `WordCloud(font_path=FONT_PATH)` |
| BoW | 문서를 단어 **개수** 표로 | `CountVectorizer` |
| TF-IDF | 문서를 단어 **중요도** 표로 | `TfidfVectorizer(min_df=5)` |
| **차이 기반 특징어** | 한 클래스에 **더 연관된 후보** | `pos_mean - neg_mean` → `argsort` |
| 어휘 자동 필터 | 너무 드문·너무 흔한 단어 잘라내기 | `min_df` · **`max_df`(고빈도어 후보 필터)** |
| n-gram | 붙어 다니는 표현 | `CountVectorizer(ngram_range=(2,2))` · 흔한 출발점 후보 **`(1,2)`** |
| 동시 출현 | 같은 문서에 함께 나오는 쌍 | `CountVectorizer(binary=True)` → `X.T @ X` |
| 관계 지도 | 단어들의 연결 구조 | 동시 출현 **네트워크**(노드=단어, 엣지=함께 나온 리뷰 수) |

- **핵심 차이는 칸의 값** — TF-IDF는 **IDF**를 곱해 흔한 단어를 깎고 기본 L2 정규화까지 적용합니다.
- **`max_df` 는 고문서빈도 단어를 거르는 후보 필터** — 문서 비율이 높은 말을 목록 없이 걸러 줍니다 (기준은 횟수가 아니라 **문서 수**). 제거된 단어를 확인한 뒤 판단해야 합니다. 실무는 **수동 도메인 불용어 + DF 필터**를 함께 씁니다.
- **워드클라우드에는 `font_path` 가 필수** — 안 넣으면 한글이 전부 □□□.
- **평균이 높은 단어 ≠ 특징어.** 양쪽에 다 나오는 공유 노이즈(`연기`·`나오`·`재미`)는 **빼기 한 번**(`pos_mean - neg_mean`)으로 저절로 걸러집니다.
- TF-IDF 벡터는 **대부분이 0인 긴 벡터**이고, **같은 단어가 나와야만** 비슷하다고 봅니다. 뜻이 같아도 **다른 말로 썼으면 남남**입니다.

## ⏭️ 예고 — 다음 단원: 텍스트 벡터화·임베딩

오늘 마지막에 확인한 한계 — "**같은 단어가 나와야 비슷하다고 본다**" — 를 넘어서는 것이 다음 단원의 주제입니다. 단어의 **글자**가 아니라 **뜻**을 숫자에 담는 방법을 배웁니다.

오늘 만든 전처리 파이프라인과 TF-IDF 는 그 출발점이자 **비교 대상**이 됩니다. 수고하셨습니다!